# CIM-Graph Library Structure

CIMantic Graphs uses the layered architecture shown below:

![cim-graph-structure](../images/cim_graph_structure.png)

### Database Layer
CIMantic Graphs supports the following files, databases and interfaces:

* CIM XML files, including models split into parts by an IEC 61970-552 metadata header (`XMLFile`)
* JSON-LD files (`JSONLDFile`)
* Blazegraph (`BlazegraphConnection`)
* GraphDB (`GraphDBConnection`)
* Neo4j (`Neo4jConnection`)
* MySQL, using the CIMTool SQL schema (`MySQLConnection`)
* RDFLib, for RDF files read into memory (`RDFlibConnection`)
* GridAPPS-D platform (`GridappsdConnection`)

All upper-level calls and routines use the same syntax, so the database can be changed by creating a different connection and setting its environment variables, with no other changes to the application. File connections need no extra packages; each database needs its driver extra (see [Installation](1_2_installation.ipynb)).

### Data Profile Layer

CIMantic Graphs supports any standard or custom CIM profile. Each UML class and attribute of the profile is a Python dataclass, which powers all of the routines and the unified API.

The built-in profiles (e.g. CIM16v33, CIM17v40, CIMHub 2023, GridAPPS-D RC4 2021, CGMES 3.0, CIM18) ship in the separate [`cimprofiles`](https://github.com/PNNL-CIM-Tools/CIM-Profiles) package, together with the `Identity` base class and the units. Custom profiles can be generated with CIMTool or Enterprise Architect Schema Composer, and several profiles can be merged at runtime. See [Profiles Overview](../02_cim_profiles/2_1_profiles_overview.ipynb).

### API Layer

Two core methods give access to any CIM class in any profile and database:

__Access to labeled property graph objects:__

* `network.graph[cim.ClassName]`: the catalog of CIM objects stored in memory, keyed by class and then by identifier (a `UUID`).

__Universal database query method:__

* `network.get_all_edges(cim.ClassName)`: gets all attributes of the objects of the class, and all objects one edge away. It works for all CIM classes, CIM profiles, serialization formats and supported databases.

Graph models also create, modify and delete objects (`create`, `modify`, `delete`), record those changes as CIM difference models, and find objects by attribute (`find_by_attribute`, `find_by_condition`).

### Knowledge Graph Layer

CIMantic Graphs offers three knowledge graph classes for different kinds of power system models:

* `BusBranchModel`: transmission bus-branch models, commonly used for planning and power flow studies
* `NodeBreakerModel`: transmission node-breaker models, commonly used in energy management systems
* `FeederModel`: distribution feeder models, with support for the single-phase unbalanced networks used in North America

Models can be centralized or distributed. A centralized model is a single labeled property graph of the network. A distributed model uses nested `DistributedArea` graphs for the equipment inside each Substation, VoltageLevel, Bay, Feeder and switch-delimited topological area of a combined T+D model. See [Graph Models](../04_graph_models/4_1_graph_models.ipynb).

### Application Layer

T+D applications access all of the power system objects through the knowledge graph, with no need to connect to the database or write custom I/O.

----

## Package Layout

| Package | Contents |
|---|---|
| `cimgraph.databases` | The connections (`XMLFile`, `JSONLDFile`, `BlazegraphConnection`, `GraphDBConnection`, `Neo4jConnection`, `MySQLConnection`, `RDFlibConnection`, `GridappsdConnection`) and their `ConnectionInterface` base class |
| `cimgraph.queries` | SPARQL, Cypher and RDFLib queries, built at runtime for any CIM profile and class |
| `cimgraph.models` | `GraphModel` and its subclasses `FeederModel`, `NodeBreakerModel`, `BusBranchModel` and `DistributedArea` |
| `cimgraph.updaters` | What changes a model: the CIM difference model builder (`incremental_builder`), GridAPPS-D difference messages (`incremental_json`), measurement value streams (`measurement_json`) and applying them to a model (`measurement_updater`) |
| `cimgraph.utils` | Shortcuts: bulk queries (`get_all_line_data`, `get_all_data`, ...), file writers (`write_xml`, `write_json_ld`) and Mermaid diagrams (`get_mermaid`, `get_mermaid_path`) |
| `cimgraph.core` | Settings from the `CIMG_*` environment variables (`get_cim_profile`, ...), per-object serialization and input validation |
| `cimprofiles` | Separate package: the CIM profiles, `Identity`, units and profile merging |

`cimgraph.data_profile` is the pre-0.6 home of the profiles. Its imports still work, with a `DeprecationWarning`: `identity`, `units` and `merge` stay for all of 1.x, while the profile packages (`cimgraph.data_profile.cim17v40` etc.) are removed in 1.0.